In [9]:
import pandas as pd
import json, gzip, gc
from pathlib import Path

RAW = Path("../data/raw")
PROC = Path("../data/processed")
PROC.mkdir(exist_ok=True, parents=True)

# ============================================================
# 1️⃣ Load refine.bio Leukemia dataset
# ============================================================
expr_path = RAW / "GSE13159_refinebio_expression.tsv"

print("🔹 Loading refine.bio expression matrix...")
refinebio_df = pd.read_csv(expr_path, sep="\t", index_col=0)
print("Shape:", refinebio_df.shape)

# Transpose if genes are rows
if refinebio_df.shape[0] > refinebio_df.shape[1]:
    refinebio_df = refinebio_df.T
    print("Transposed refine.bio → samples × genes:", refinebio_df.shape)

# Create placeholder metadata
samples = refinebio_df.index.tolist()
meta_df = pd.DataFrame({
    "sample": samples,
    "disease": ["UNKNOWN"] * len(samples)
})
print(f"Metadata placeholder: {meta_df.shape}")

# ✅ REMOVE VERSION NUMBERS FROM LEUKEMIA ENSEMBL IDs
refinebio_df.columns = refinebio_df.columns.str.split('.').str[0]
print("Leukemia genes after removing versions:", refinebio_df.shape[1])

gc.collect()

# ============================================================
# 2️⃣ Load TCGA + SIMPLE ENSEMBL ID MATCHING
# ============================================================
tcga_path = RAW / "tcga_RSEM_gene_tpm.gz"

print("\n🔹 Loading TCGA expression data...")
with gzip.open(tcga_path, "rt") as f:
    tcga_df = pd.read_csv(f, sep="\t", index_col=0)
print("Raw TCGA shape:", tcga_df.shape)

# ✅ SIMPLE FIX: Remove version numbers from TCGA Ensembl IDs
tcga_df.index = tcga_df.index.str.split('.').str[0]
print("TCGA genes after removing versions:", tcga_df.shape[0])

# Remove duplicates
tcga_df = tcga_df[~tcga_df.index.duplicated(keep='first')]
print("TCGA after deduplication:", tcga_df.shape)

gc.collect()

# ============================================================
# 3️⃣ Intersect & align both datasets
# ============================================================
overlap_genes = refinebio_df.columns.intersection(tcga_df.index)
print(f"\n🔹 Found {len(overlap_genes)} overlapping genes.")

if len(overlap_genes) > 0:
    leuk_expr = refinebio_df[overlap_genes].astype("float32")
    tcga_expr = tcga_df.loc[overlap_genes].T.astype("float32")

    print(f"✅ Common genes: {len(overlap_genes)} / {len(refinebio_df.columns)} "
          f"({100*len(overlap_genes)/len(refinebio_df.columns):.2f}% retained)")

    # ============================================================
    # 4️⃣ Save aligned outputs
    # ============================================================
    leuk_expr.to_csv(PROC / "leukemia_filtered.csv", float_format="%.6f")
    tcga_expr.to_csv(PROC / "tcga_filtered.csv", float_format="%.6f")
    meta_df.to_csv(PROC / "leukemia_sample_info.csv", index=False)

    print("\n💾 Saved processed data to data/processed/")
    print("TCGA shape:", tcga_expr.shape)
    print("Leukemia shape:", leuk_expr.shape)
    print("Sample info shape:", meta_df.shape)
else:
    print("❌ ERROR: Still no overlapping genes!")
    print("TCGA genes sample:", tcga_df.index[:10].tolist())
    print("Leukemia genes sample:", refinebio_df.columns[:10].tolist())

gc.collect()

🔹 Loading refine.bio expression matrix...
Shape: (20056, 357)
Transposed refine.bio → samples × genes: (357, 20056)
Metadata placeholder: (357, 2)
Leukemia genes after removing versions: 20056

🔹 Loading TCGA expression data...
Raw TCGA shape: (60498, 10535)
TCGA genes after removing versions: 60498
TCGA after deduplication: (60498, 10535)

🔹 Found 20034 overlapping genes.
✅ Common genes: 20034 / 20056 (99.89% retained)

💾 Saved processed data to data/processed/
TCGA shape: (10535, 20034)
Leukemia shape: (357, 20034)
Sample info shape: (357, 2)


0

In [ ]:
import GEOparse
import pandas as pd
import os

# Make sure the directory exists
os.makedirs('./data/raw/', exist_ok=True)

print("🔹 Downloading GEO metadata (brief version)...")
# This should download the small metadata file
gse = GEOparse.get_GEO(geo="GSE13159", destdir='./data/raw/', how='brief')

print(f"🔹 Loaded {len(gse.gsms)} samples")

# Extract labels from metadata
labels = []
for gsm_name, gsm in gse.gsms.items():
    metadata = gsm.metadata
    
    # Look for disease info in characteristics
    disease = "UNKNOWN"
    if 'characteristics_ch1' in metadata:
        chars = ' '.join(metadata['characteristics_ch1']).lower()
        if 'aml' in chars or 'acute myeloid' in chars:
            disease = 'AML'
        elif 'all' in chars or 'acute lymphoblastic' in chars:
            disease = 'ALL' 
        elif 'cml' in chars or 'chronic myeloid' in chars:
            disease = 'CML'
        elif 'cll' in chars or 'chronic lymphocytic' in chars:
            disease = 'CLL'
        elif 'healthy' in chars or 'normal' in chars:
            disease = 'Healthy'
    
    labels.append({'sample': gsm_name, 'disease': disease})

# Convert to DataFrame and save
labels_df = pd.DataFrame(labels)
labels_df.to_csv('./data/processed/leukemia_sample_info.csv', index=False)

print(f"✅ Extracted labels for {len(labels_df)} samples")
print("Label distribution:")
print(labels_df['disease'].value_counts())


# Filter labels to only our 357 samples
leukemia_expr_df = pd.read_csv('./data/processed/leukemia_filtered.csv', index_col=0)
our_samples = leukemia_expr_df.index.tolist()

filtered_labels = labels_df[labels_df['sample'].isin(our_samples)]
filtered_labels.to_csv('./data/processed/leukemia_sample_info.csv', index=False)

print(f"✅ Filtered to {len(filtered_labels)} samples we actually have")
print("Final label distribution:")
print(filtered_labels['disease'].value_counts())